# D1.2 · Context that makes triage work

**Function D — Security Operations → The SOC Analyst & Detection Engineer**  ·  *AI for Security*

---

**Risk.** Generic triage agents underperform your worst analyst.

**Control.** Feed the baseline, known FPs, crown-jewel map and prior decisions.

**This lab.** Show a context-loaded triage loop beating a generic one.

| | |
|---|---|
| Open-source tooling | Wazuh |
| Open-weight models | GLM-4.6, Llama 3.3 |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("D1.2"))

Context is what makes triage work, and for agent alerts the context that matters is identity: who acted, through whom, holding what.

In [ ]:
from cybercommons import identity, soc
import time

alice = identity.mint("alice")
patch = identity.exchange(alice, "patch-agent", {"repo:read", "repo:write"})

alert_event = soc.Event(time.time(), "patch-agent", "read_file", "/work/.env")

print("alert without context:")
print(f"   {alert_event.actor} read {alert_event.target}")
print("\nalert with identity context:")
print(f"   actor        {patch.actor}")
print(f"   on behalf of {patch.sub}")
print(f"   chain        {' → '.join(patch.chain())}")
print(f"   scopes       {sorted(patch.scopes)}")
print(f"   token fp     {patch.fingerprint()}")
print("\nThe second version answers 'is this expected?' — the first cannot.")

The scope list is the decisive field. Reading `.env` is alarming for an agent scoped to `repo:read`; for a secrets-rotation agent it is Tuesday. Without scope in the alert, every analyst has to guess.

### Expect

The bare alert shows actor and target only. The enriched version adds the principal, the full delegation chain, the held scopes and a token fingerprint.

### Your turn

List the five fields your agent alerts would need to be triageable without a second lookup. Then check how many your telemetry actually carries.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/D1.2.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*